# Shor's Algorithm — Amazon Braket

Shor's algorithm factors an integer $N$ by:

1. Choosing random $a < N$ with $\gcd(a, N) = 1$.
2. Using quantum **order-finding** to find the order $r$ of $a \bmod N$.
3. Computing $\gcd(a^{r/2} \pm 1, N)$ to extract factors.

This notebook demonstrates the key ideas on $N = 15$.
A full quantum circuit requires many qubits; we show the
classical post-processing and circuit structure.

In [ ]:
import math

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Order-finding (classical)

Find $r$ such that $a^r \equiv 1 \pmod{N}$.

In [ ]:
N = 15
a = 7

print(f"N = {N}, a = {a}")
print()

r = 1
val = a % N
while val != 1:
    val = (val * a) % N
    r += 1

print(f"  7^1 mod 15 = {7 % 15}")
print(f"  7^2 mod 15 = {49 % 15}")
print(f"  7^3 mod 15 = {343 % 15}")
print(f"  7^4 mod 15 = {2401 % 15}  <- 1, so order r = {r}")
print()
print(f"gcd(7^2 - 1, 15) = gcd({7**2 - 1}, 15) = {math.gcd(7**2 - 1, 15)}")
print(f"gcd(7^2 + 1, 15) = gcd({7**2 + 1}, 15) = {math.gcd(7**2 + 1, 15)}")
print(f"Factors of 15: {math.gcd(7**2 - 1, 15)} x {math.gcd(7**2 + 1, 15)} = 15")

## Order-finding circuit structure

The quantum circuit uses precision qubits + a target register:

1. Hadamard on precision qubits.
2. Controlled modular exponentiation: $C\text{-}U^{2^k}$.
3. Inverse QFT on precision register.
4. Measurement.

In [ ]:
n_precision = 3
circuit = Circuit()
for i in range(n_precision):
    circuit.h(i)

print("Precision register (Hadamards only):")
print(circuit)
print()
print("Full Shor would add controlled modular exponentiation:")
print("  C-U^1: controlled-(7^1 mod 15)  on 4 target qubits")
print("  C-U^2: controlled-(7^2 mod 15)")
print("  C-U^4: controlled-(7^4 mod 15)")
print("Then inverse QFT + measurement.")

## Simulated measurement outcomes

With $r = 4$, the precision register peaks at multiples of $2^n / r$.

In [ ]:
n = 3
N_states = 8
r = 4

counts = {}
for j in range(N_states):
    bits = format(j, f'0{n}b')
    counts[bits] = 250 if j % (N_states // r) == 0 else 1

print(f"Simulated counts: {counts}")
print()
print("Peaks at |000>, |010>, |100>, |110> (multiples of 2).")
print("From |010>: phi = 2/8 = 0.25 -> continued fraction -> r = 4.")

## Full Shor (classical simulation)

In [ ]:
import random
random.seed(42)

N = 15
a = random.randrange(2, N)
while math.gcd(a, N) != 1:
    a = random.randrange(2, N)

print(f"Chosen a = {a}, gcd({a}, {N}) = {math.gcd(a, N)}")

r = 1
val = a % N
while val != 1:
    val = (val * a) % N
    r += 1

print(f"Order r = {r}")

x = pow(a, r // 2, N)
f1 = math.gcd(x - 1, N)
f2 = math.gcd(x + 1, N)

print(f"a^(r/2) mod N = {x}")
print(f"gcd({x} - 1, {N}) = {f1}")
print(f"gcd({x} + 1, {N}) = {f2}")
print(f"Factors of {N}: {f1} x {f2} = {f1 * f2}")